In [ ]:
!pip install -q datasets

from datasets import load_dataset, DatasetDict
import re
import math


In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted


In [ ]:
# Load original Ashaar dataset (train split only)
raw_ds = load_dataset("arbml/ashaar", split="train")
print(raw_ds)
print(raw_ds.features)


Dataset({
    features: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type'],
    num_rows: 254630
})
{'poem title': Value('string'), 'poem meter': Value('string'), 'poem verses': List(Value('string')), 'poem theme': Value('string'), 'poem url': Value('string'), 'poet name': Value('string'), 'poet description': Value('string'), 'poet url': Value('string'), 'poet era': Value('string'), 'poet location': Value('string'), 'poem description': List({'attributes': {'class': Value('string'), 'color': Value('string'), 'dir': Value('string'), 'face': Value('string'), 'id': Value('string'), 'lang': Value('string'), 'style': Value('string')}, 'children': List({'attributes': {'color': Value('string'), 'dir': Value('string'), 'face': Value('string'), 'href': Value('string'), 'id': Value('string'), 'lang': Value('string'), 'style': Value('string'), 'title': Value('strin

In [ ]:
# Regexes for noise in verses
latin_re = re.compile(r"[A-Za-z]")
digit_re = re.compile(r"[0-9٠-٩]")  # Arabic + Latin digits
url_re   = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)


In [ ]:
# --- FULL DROP LIST FOR "poem meter" ---
drop_meters = {
    # Explicit deletions you listed first
    "بحر التفعيلة",
    "الكان كان",
    "اللويحاني",
    "بحر التفعيله",
    "بحر تفعيلة الرجز",
    "بحر تفعيلة الرمل",
    "بحر تفعيلة الكامل",
    "بحر تفعيلة المتقارب",
    "بحر مشطور الرجز",
    "بحر مشطور السريع",
    "بحر مشطور الطويل",
    "شعر التفعيلة",
    "شعر حر",
    "عامي",
    "عدة أبحر",
    "عموديه",
    "نثرية",
    "نثريه",
    "زجل",
    "التفعيلة",
    "التفعيله",

    # Rare meters block
    "بحر مجزوء المتقارب",
    "المقتضب",
    "بحر مشطور الرجز",
    "بحر مجزوء موشح",
    "بحر منهوك المنسرح",
    "بحر المقتضب",
    "السلسلة",
    "بحر مجزوء الطويل",
    "بحر مجزوء السريع",
    "بحر مجزوء الدوبيت",
    "بحر التفعيله",
    "بحر منهوك الرجز",
    "بحر مجزوء المتدارك",
    "بحر مربع الرجز",
    "بحر السلسلة",
    "بحر مخلع الرمل",
    "بحر مخلع الكامل",
    "بحر مجزوء المديد",
    "الهجيني",
    "الكان كان",
    "بحر مجزوء الهزج",
    "بحر مخلع موشح",
    "بحر الكامل المقطوع",
    "بحر تفعيلة الكامل",
    "بحر مجزوء المجتث",
    "بحر مربع البسيط",
    "بحر المتدارك المنهوك",
    "بحر تفعيلة الرمل",
    "بحر تفعيلة الرجز",
    "بحر تفعيلة المتقارب",
    "بحر مجزوء الرمل",
    "بحر القوما",
    "بحر الخبب",
    "بحر مجزوء المنسرح",
    "بحر مجزوء المواليا",
    "بحر مشطور الطويل",
    "بحر أحذ المديد",
    "بحر مخلع السريع",
    "بحر مشطور السريع",
    "بحر أحذ الوافر",
    "بحر منهوك البسيط",
    "بحر منهوك الكامل",
    "بحر مخلع الرجز",
    "زجل",
    "الصخري",
    "الحداء",
}
# Strip any accidental spaces we may have in this set itself
drop_meters = {m.strip() for m in drop_meters}


In [ ]:
def has_valid_meter_and_verses(ex):
    m = ex["poem meter"]
    if m is None or str(m).strip() == "":
        return False

    verses = ex["poem verses"]
    if verses is None:
        return False
    if not isinstance(verses, list) or len(verses) == 0:
        return False

    # At least one non-empty string in the list
    for v in verses:
        if isinstance(v, str) and v.strip() != "":
            return True
    return False

ds = raw_ds.filter(has_valid_meter_and_verses)
print("After meter+verses non-empty filter:", len(ds))


After meter+verses non-empty filter: 153353


In [ ]:
def keep_not_dropped_meter(ex):
    m = ex["poem meter"]
    name = str(m).strip()
    return name not in drop_meters

ds = ds.filter(keep_not_dropped_meter)
print("After dropping banned meters:", len(ds))


Filter:   0%|          | 0/153353 [00:00<?, ? examples/s]

After dropping banned meters: 146045


In [ ]:
bad_langs = {"-", "شعبي", "عامي"}

def keep_lang(ex):
    lang = ex["poem language type"]
    if lang is None:
        return True  # you said: for other categories, you tolerate empties
    return str(lang).strip() not in bad_langs

ds = ds.filter(keep_lang)
print("After language filtering:", len(ds))


Filter:   0%|          | 0/146045 [00:00<?, ? examples/s]

After language filtering: 145962


In [ ]:
def verses_clean_of_noise(ex):
    verses = ex["poem verses"] or []
    for v in verses:
        if not isinstance(v, str):
            continue
        txt = v
        if latin_re.search(txt) or digit_re.search(txt) or url_re.search(txt):
            return False
    return True

ds = ds.filter(verses_clean_of_noise)
print("After removing verses with latin/digits/urls:", len(ds))


Filter:   0%|          | 0/145962 [00:00<?, ? examples/s]

After removing verses with latin/digits/urls: 145904


In [ ]:
def add_num_verses(ex):
    verses = ex["poem verses"] or []
    n_hemistics = len(verses)
    n_verses = math.ceil(n_hemistics / 2.0)
    ex["num_verses"] = int(n_verses)
    return ex

ds = ds.map(add_num_verses)
print(ds)

# Filter by maximum allowed number of verses
MAX_VERSES = 110

def keep_short_poem(ex):
    return ex["num_verses"] <= MAX_VERSES

ds = ds.filter(keep_short_poem)
print("After dropping poems with > 110 verses:", len(ds))


Map:   0%|          | 0/145904 [00:00<?, ? examples/s]

Dataset({
    features: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type', 'num_verses'],
    num_rows: 145904
})


Filter:   0%|          | 0/145904 [00:00<?, ? examples/s]

After dropping poems with > 110 verses: 145167


In [ ]:
def normalize_location(ex):
    loc = ex["poet location"]
    if loc == "سورية":
        ex["poet location"] = "سوريا"
    return ex

ds = ds.map(normalize_location)


Map:   0%|          | 0/145167 [00:00<?, ? examples/s]

In [ ]:
def clean_meter(ex):
    m = ex["poem meter"]
    if m is None:
        return ex
    s = str(m).strip()

    # Specific replacements
    if s == "بسيط":
        s = "البسيط"
    if s == "بحر موشح":
        s = "الموشح"

    # Remove leading "بحر " or "البحر "
    if s.startswith("بحر "):
        s = s[4:]  # remove the first 4 chars "بحر "
    elif s.startswith("البحر "):
        s = s[6:]  # remove "البحر "

    s = s.strip()  # final trim

    ex["poem meter"] = s
    return ex

ds = ds.map(clean_meter)


Map:   0%|          | 0/145167 [00:00<?, ? examples/s]

In [ ]:
def add_hemistic_tags(ex):
    verses = ex["poem verses"] or []
    tagged = []
    for i, v in enumerate(verses):
        if not isinstance(v, str):
            v = "" if v is None else str(v)
        if i % 2 == 0:
            # first hemistic of verse
            tagged.append(v + "<s>")
        else:
            # second hemistic of verse
            tagged.append(v + "<a>")
    ex["poem verses"] = tagged
    return ex

ds = ds.map(add_hemistic_tags)


Map:   0%|          | 0/145167 [00:00<?, ? examples/s]

In [ ]:
def add_poem_id(ex, idx):
    ex["poem id"] = int(idx)  # 0,1,2,...
    return ex

ds = ds.map(add_poem_id, with_indices=True)
print(ds)


Map:   0%|          | 0/145167 [00:00<?, ? examples/s]

Dataset({
    features: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type', 'num_verses', 'poem id'],
    num_rows: 145167
})


In [ ]:
clean_dd = DatasetDict({"train": ds})

clean_dd.push_to_hub("Shaer-AI/ashaar-preprocessed")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/146 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   2%|1         | 2.61MB /  137MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Shaer-AI/ashaar-preprocessed/commit/d119e94dc8231297f63e97d037b21bcb3f245cff', commit_message='Upload dataset', commit_description='', oid='d119e94dc8231297f63e97d037b21bcb3f245cff', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Shaer-AI/ashaar-preprocessed', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Shaer-AI/ashaar-preprocessed'), pr_revision=None, pr_num=None)

In [ ]:
import pandas as pd

df = ds.to_pandas()
print("Total number of instances (rows):", len(df))
print("\n=== Missing values per feature ===")
missing_counts = df.isna().sum()
missing_frac   = df.isna().mean()
missing_summary = pd.DataFrame({
    "n_missing": missing_counts,
    "frac_missing": missing_frac
}).sort_values("n_missing", ascending=False)
print(missing_summary)


Total number of instances (rows): 145167

=== Missing values per feature ===
                    n_missing  frac_missing
poem description       132027      0.909484
poet location           92038      0.634015
poem theme              84597      0.582756
poet description        84594      0.582736
poem title              84470      0.581882
poem language type      60697      0.418118
poet era                12271      0.084530
poet url                  165      0.001137
poem url                   38      0.000262
poem meter                  0      0.000000
poet name                   0      0.000000
poem verses                 0      0.000000
num_verses                  0      0.000000
poem id                     0      0.000000


In [ ]:
cols_of_interest = [
    "poem meter",
    "poem theme",
    "poet era",
    "poet location",
    "poem language type",
]

for col in cols_of_interest:
    print("\n" + "="*80)
    print(f"COLUMN: {col}")
    print("="*80)

    n_missing = df[col].isna().sum()
    n_unique = df[col].nunique(dropna=True)
    print(f"Missing values: {n_missing}")
    print(f"Unique non-null values: {n_unique}")

    print("\nValue counts:")
    print(df[col].value_counts(dropna=False))



COLUMN: poem meter
Missing values: 0
Unique non-null values: 25

Value counts:
poem meter
الطويل          34400
الكامل          24930
البسيط          20423
الوافر          13192
الخفيف          11915
السريع           7937
المتقارب         6162
الرجز            5972
الرمل            4750
المنسرح          3155
المجتث           2348
مجزوء الكامل     2260
الموشح           1625
الدوبيت          1064
مجزوء الرجز       779
الهزج             764
مخلع البسيط       744
المديد            689
أحذ الكامل        478
المواليا          433
مجزوء الخفيف      360
مجزوء الوافر      319
المتدارك          234
مجزوء البسيط      176
المضارع            58
Name: count, dtype: int64

COLUMN: poem theme
Missing values: 84597
Unique non-null values: 18

Value counts:
poem theme
None              84597
قصيدة قصيره       24690
قصيدة عامه        16934
قصيدة مدح          4813
قصيدة رومنسيه      3636
قصيدة حزينه        1923
قصيدة عتاب         1860
قصيدة هجاء         1537
قصيدة غزل          1252
قصيدة دينية        103

In [ ]:
# 1) No empty poem meter
assert df["poem meter"].isna().sum() == 0
assert (~df["poem meter"].astype(str).str.strip().eq("")).all()

# 2) No banned language types
bad_langs = {"-", "شعبي", "عامي"}
assert not df["poem language type"].isin(bad_langs).any()


# 3) No banned meters
assert not df["poem meter"].isin(drop_meters).any()

# 4) No poems with num_verses > 110
assert (df["num_verses"] <= 110).all()

# 5) سورية → سوريا normalization
if "سورية" in df["poet location"].dropna().unique():
    print("WARNING: 'سورية' still present in poet location!")
else:
    print("OK: سورية successfully normalized to سوريا")

has_noise = df.apply(check_noise_in_verses, axis=1).any()
print("Any poem with latin/digits/urls still present in verses? ->", has_noise)


OK: سورية successfully normalized to سوريا
Any poem with latin/digits/urls still present in verses? -> True
